import os
import json

# 替换目标
old_label = 'bolt23'
new_label = 'bolt17'

# 目标文件夹
json_dirs = {
    'train': '/root/autodl-tmp/coco_dataset/jsons/train',
    'val': '/root/autodl-tmp/coco_dataset/jsons/val'
}

total_files_modified = 0
total_objects_modified = 0

print('\n======= 🔧 批量处理单图 JSON 标注文件（jsons/train 与 val） =======')

for split, folder in json_dirs.items():
    if not os.path.exists(folder):
        print(f"❌ 文件夹不存在: {folder}")
        continue

    json_files = [f for f in os.listdir(folder) if f.endswith('.json')]
    modified_file_count = 0
    modified_obj_count = 0

    for json_file in json_files:
        path = os.path.join(folder, json_file)

        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 通常单图标注结构为 'shapes' 或 'annotations'
        annotations = data.get('annotations', data.get('shapes', []))

        modified = False
        for ann in annotations:
            label = ann.get('label') or ann.get('name')  # 支持不同结构
            if label == old_label:
                if 'label' in ann:
                    ann['label'] = new_label
                if 'name' in ann:
                    ann['name'] = new_label
                modified = True
                modified_obj_count += 1

        if modified:
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            modified_file_count += 1

    print(f"✅ {split}: 修改 {modified_file_count} 个文件，共替换 {modified_obj_count} 个标注")
    total_files_modified += modified_file_count
    total_objects_modified += modified_obj_count

# 总结
print('\n======= ✅ 全部处理完成 =======')
print(f'📄 修改的 JSON 文件数：{total_files_modified}')
print(f'🔧 替换的标注框总数：{total_objects_modified}')
